## Structured Streaming Fundamentals

Most data pipelines in Spark work in batches — read a file, transform it, write the result. But the real world produces data continuously: card transactions arriving every millisecond, ATM withdrawals, loan repayments, fraud alerts. Waiting for a daily batch means decisions are always stale. **Structured Streaming** brings Spark's DataFrame API to continuous, unbounded data.

The core idea is elegant: **treat the incoming stream as an infinite table**. Every new record appended to the stream is a new row appended to that table. You write a query against that table — the same transformations, filters, aggregations, and joins you already know — and Spark runs it continuously as new data arrives.

```text
Input Stream (unbounded table)
──────────────────────────────────────────────────────────────►  time
  row 1 │ row 2 │ row 3 │ row 4 │ row 5 │ row 6 │ row 7 │ ...

Your query runs continuously on new rows → Output Table
```

In this notebook you will learn:
- The **unbounded table model** and how Spark processes micro-batches
- How to read a stream with `readStream`
- **Output modes**: append, update, complete
- **Triggers**: how often Spark checks for new data
- Writing to sinks with `writeStream`
- Monitoring a running stream with `StreamingQuery`

### The Unbounded Table Model

Spark Structured Streaming represents an input stream as a table that grows indefinitely. Every trigger interval, Spark processes a **micro-batch** — the new rows that arrived since the last trigger. The results are written to an **output table** (or sink) according to the **output mode**.

```text
┌──────────────────────────────────────────────────────────┐
│                    Input Stream Table                    │
│  txn_id  │ amount │ category  │ txn_ts                  │
│ ─────────┼────────┼───────────┼──────────────────────── │
│  T001    │  120   │ FOOD      │ 2024-01-01 09:00:01     │
│  T002    │  450   │ TRAVEL    │ 2024-01-01 09:00:02     │
│  T003    │   80   │ FOOD      │ 2024-01-01 09:00:05     │
│  T004    │  200   │ SHOPPING  │ 2024-01-01 09:00:07     │
│  ...     │  ...   │ ...       │ ...         (growing)   │
└──────────────────────────────────────────────────────────┘
                          │
              Your Streaming Query
              (filter, groupBy, agg)
                          │
                          ▼
┌──────────────────────────────────────────────────────────┐
│                    Output Table / Sink                   │
│  category  │ total_spend │ last updated                 │
│ ───────────┼─────────────┼───────────────────────────── │
│  FOOD      │    200      │ updated each micro-batch      │
│  TRAVEL    │    450      │                               │
│  SHOPPING  │    200      │                               │
└──────────────────────────────────────────────────────────┘
```

**Key vocabulary:**

| Term | Meaning |
|---|---|
| **Source** | Where data comes from: files, Kafka, Rate source, socket |
| **Micro-batch** | One chunk of new data processed in a single Spark job |
| **Trigger** | How often Spark checks for new data |
| **Output mode** | Which rows are written to the sink each trigger |
| **Sink** | Where results are written: memory, files, Kafka, Delta |
| **Checkpoint** | State and progress saved to disk so the stream can restart |
| **Watermark** | How long to wait for late-arriving events before closing a window |

In [ ]:
import os
import time
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, max as _max,
    window, current_timestamp, expr, to_timestamp,
    lit, concat, rand, when, floor
)

spark = (
    SparkSession.builder
    .appName("StructuredStreamingFundamentals")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.streaming.schemaInference", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BASE = os.path.dirname(os.path.abspath("15-structured-streaming-fundamentals.ipynb"))
DATA = os.path.join(BASE, "data")

# Scratch directories used by streaming examples — cleaned up at the end
STREAM_INPUT  = os.path.join(BASE, "_stream_input")
STREAM_OUTPUT = os.path.join(BASE, "_stream_output")
CHECKPOINT    = os.path.join(BASE, "_checkpoints")

for d in [STREAM_INPUT, STREAM_OUTPUT, CHECKPOINT]:
    os.makedirs(d, exist_ok=True)

print("SparkSession ready")
print(f"Spark version : {spark.version}")

### Source 1 — The Rate Source (Built-in Generator)

The **rate source** is the simplest way to explore streaming without any external system. It generates rows at a fixed rate with two columns: `timestamp` (event time) and `value` (monotonically increasing integer). It is available in all Spark environments with no extra setup.

Use the rate source to:
- Understand how `readStream` and `writeStream` work
- Test aggregations and window functions before connecting to Kafka
- Benchmark cluster throughput

In [ ]:
# Read from the rate source — generates 5 rows per second
rate_stream = (
    spark
    .readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

print("isStreaming:", rate_stream.isStreaming)   # True — this is a streaming DataFrame
rate_stream.printSchema()
# timestamp: TimestampType — the event time of each generated row
# value    : LongType     — 0, 1, 2, 3, ... (monotonically increasing)

### Writing to a Sink — writeStream

`writeStream` starts the streaming query. The key options:

| Option | What it controls |
|---|---|
| `.format("memory")` | In-memory sink — results queryable as a temp view (dev/testing only) |
| `.format("console")` | Print to stdout each micro-batch (dev/testing only) |
| `.format("parquet")` | Write to Parquet files (production use) |
| `.format("delta")` | Write to Delta Lake (production use) |
| `.format("kafka")` | Publish to a Kafka topic |
| `.queryName("name")` | Name the query — used to find it in `spark.streams.active` |
| `.outputMode("append")` | Which rows to emit each trigger (see next section) |
| `.trigger(...)` | How often to run a micro-batch |
| `.checkpointLocation(path)` | Where to save query state so it can restart |
| `.start()` | **Actually starts the query** — returns a `StreamingQuery` handle |

In [ ]:
from pyspark.sql.streaming import StreamingQuery

# Write the rate stream to the memory sink — results land in a temp view called "rate_sink"
rate_query: StreamingQuery = (
    rate_stream
    .writeStream
    .format("memory")
    .queryName("rate_sink")
    .outputMode("append")
    .start()
)

print("Query name   :", rate_query.name)
print("Query id     :", rate_query.id)
print("isActive     :", rate_query.isActive)

# Let the stream run for a few seconds to accumulate some rows
time.sleep(4)

# Query the memory sink like a regular table
spark.sql("SELECT * FROM rate_sink ORDER BY value LIMIT 10").show()
print(f"Total rows so far: {spark.sql('SELECT count(*) FROM rate_sink').collect()[0][0]}")

In [ ]:
# Inspect the last progress report — Spark updates this after every micro-batch
import json
if rate_query.lastProgress:
    progress = rate_query.lastProgress
    print(f"Batch id           : {progress['batchId']}")
    print(f"Input rows/sec     : {progress['inputRowsPerSecond']}")
    print(f"Processed rows/sec : {progress['processedRowsPerSecond']}")
    print(f"Trigger duration   : {progress['triggerExecution']} ms")

# List all active streaming queries in this SparkSession
print("\nActive queries:")
for q in spark.streams.active:
    print(f"  {q.name}  isActive={q.isActive}")

rate_query.stop()
print("\nQuery stopped. isActive:", rate_query.isActive)

### Source 2 — File Source (Auto-Discover New Files)

The **file source** watches a directory and processes each new file as it appears. Spark tracks which files have already been processed (using its checkpoint) and only reads new ones. Supported formats: CSV, JSON, Parquet, ORC, Delta.

This is the most common streaming source outside Kafka — for example, an upstream process writing one JSON file per minute of transaction data.

**Schema must be provided explicitly** — Spark cannot infer schema for streaming file sources (it has no guarantee that any file exists when the query starts).

In [ ]:
# Schema for incoming transaction files
txn_schema = StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("card_id",     StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])

# Read streaming from a directory — any new JSON file dropped here is processed
file_stream = (
    spark
    .readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 1)   # process one file per micro-batch
    .json(STREAM_INPUT)
)

print("isStreaming:", file_stream.isStreaming)
file_stream.printSchema()

In [ ]:
import json as _json
from decimal import Decimal
from datetime import datetime, timedelta
import random, string

CATEGORIES = ["FOOD", "TRAVEL", "SHOPPING", "UTILITIES", "ENTERTAINMENT"]
STATUSES   = ["SUCCESS", "SUCCESS", "SUCCESS", "DECLINED", "PENDING"]

def write_txn_file(file_index: int, num_rows: int = 20):
    """Write a batch of synthetic transactions as a JSON file to STREAM_INPUT."""
    rows = []
    base_ts = datetime(2024, 1, 1, 9, 0, 0) + timedelta(minutes=file_index * 5)
    for i in range(num_rows):
        rows.append({
            "txn_id":      f"T{file_index:03d}{i:04d}",
            "card_id":     f"CARD{''.join(random.choices(string.digits, k=4))}",
            "customer_id": f"CUST{random.randint(1, 10):04d}",
            "amount":      round(random.uniform(10, 500), 2),
            "category":    random.choice(CATEGORIES),
            "txn_ts":      (base_ts + timedelta(seconds=i)).strftime("%Y-%m-%dT%H:%M:%S"),
            "status":      random.choice(STATUSES),
            "is_fraud":    random.random() < 0.05,
        })
    path = os.path.join(STREAM_INPUT, f"batch_{file_index:03d}.json")
    with open(path, "w") as f:
        for row in rows:
            f.write(_json.dumps(row) + "\n")
    print(f"Written: {path}  ({num_rows} rows)")

# Write the first batch before starting the stream
write_txn_file(0)
write_txn_file(1)

In [ ]:
# Simple pass-through: write each incoming row to the memory sink
file_query = (
    file_stream
    .writeStream
    .format("memory")
    .queryName("txn_stream")
    .outputMode("append")
    .option("checkpointLocation", os.path.join(CHECKPOINT, "file_query"))
    .start()
)

time.sleep(6)  # let the first two files be processed

spark.sql("SELECT * FROM txn_stream ORDER BY txn_ts LIMIT 10").show(truncate=False)
print(f"Rows ingested so far: {spark.sql('SELECT count(*) FROM txn_stream').collect()[0][0]}")

In [ ]:
# Drop a new file — Spark will pick it up in the next micro-batch
write_txn_file(2)
time.sleep(6)

print(f"Rows after new file: {spark.sql('SELECT count(*) FROM txn_stream').collect()[0][0]}")
file_query.stop()

### Output Modes

The output mode controls **which rows** Spark writes to the sink in each micro-batch:

| Mode | Rows emitted per trigger | Use when |
|---|---|---|
| **append** | Only new rows added since the last trigger | No aggregation, or append-only aggregations with watermarks |
| **update** | Only rows that changed since the last trigger | Aggregations where you want incremental updates |
| **complete** | The entire result table, rewritten each trigger | Small aggregations (counts, totals) written to an in-memory or overwrite sink |

```text
Trigger 1 result:  category=FOOD, count=3
Trigger 2 result:  category=FOOD, count=5   ← 2 more FOOD rows arrived

  append mode   →  writes (FOOD, 5) ... but INCORRECT for aggregations (no dedup)
  update mode   →  writes only (FOOD, 5) — the updated row
  complete mode →  writes the full table: (FOOD,5), (TRAVEL,2), (SHOPPING,1), ...
```

**Rules:**
- `append` is the default and works for stateless operations (filter, select, map)
- `complete` requires an aggregation — Spark must hold the full result in state
- `update` requires an aggregation; only unsupported for some sinks (e.g., parquet files)

In [ ]:
# Re-read the already-written files as a fresh stream
txn_stream = (
    spark
    .readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 1)
    .json(STREAM_INPUT)
)

# Aggregation: count transactions per category — use COMPLETE mode
category_counts = (
    txn_stream
    .groupBy("category")
    .agg(
        count("txn_id").alias("txn_count"),
        _sum("amount").alias("total_amount"),
    )
)

complete_query = (
    category_counts
    .writeStream
    .format("memory")
    .queryName("category_complete")
    .outputMode("complete")   # full result table written every trigger
    .option("checkpointLocation", os.path.join(CHECKPOINT, "complete_query"))
    .start()
)

time.sleep(10)
print("=== COMPLETE mode — full result table after each trigger ===")
spark.sql("SELECT * FROM category_complete ORDER BY total_amount DESC").show()

complete_query.stop()

In [ ]:
# UPDATE mode — only emit rows that changed since last trigger
txn_stream2 = (
    spark
    .readStream
    .schema(txn_schema)
    .option("maxFilesPerTrigger", 1)
    .json(STREAM_INPUT)
)

category_agg2 = (
    txn_stream2
    .groupBy("category")
    .agg(count("txn_id").alias("txn_count"))
)

update_query = (
    category_agg2
    .writeStream
    .format("memory")
    .queryName("category_update")
    .outputMode("update")   # only changed rows per trigger
    .option("checkpointLocation", os.path.join(CHECKPOINT, "update_query"))
    .start()
)

time.sleep(10)
print("=== UPDATE mode — cumulative state (memory sink accumulates changed rows) ===")
spark.sql("SELECT * FROM category_update ORDER BY txn_count DESC").show()

update_query.stop()

### Triggers — Controlling Micro-Batch Cadence

A **trigger** defines how often Spark checks for new data and runs a micro-batch.

| Trigger | API | Behaviour |
|---|---|---|
| **Default** (unspecified) | `.trigger(processingTime="0 seconds")` | Run next batch as soon as previous one finishes |
| **Fixed interval** | `.trigger(processingTime="30 seconds")` | Run every 30 s; skip if previous batch is still running |
| **Once** | `.trigger(once=True)` | Process all available data in **one** batch, then stop. Useful for scheduled jobs |
| **Available Now** | `.trigger(availableNow=True)` | Process all available data in **multiple** optimally-sized batches, then stop |
| **Continuous** | `.trigger(continuous="1 second")` | Experimental low-latency mode; asynchronous checkpointing, ~1 ms latency |

**Choosing a trigger interval:**
- Lower interval → lower end-to-end latency, more small Spark jobs (higher overhead per row)
- Higher interval → higher latency, fewer larger Spark jobs (more efficient per row)
- `once` / `availableNow` turn Structured Streaming into a scheduled micro-batch job — perfect for hourly pipelines that still need exactly-once semantics and offset tracking

In [ ]:
from pyspark.sql.streaming import StreamingQuery
from pyspark.sql import functions as F

# Write two more input files before the once-trigger query runs
write_txn_file(3)
write_txn_file(4)

txn_stream3 = (
    spark
    .readStream
    .schema(txn_schema)
    .json(STREAM_INPUT)
)

# Trigger(once=True) — process ALL available files in one shot, then stop
once_query = (
    txn_stream3
    .filter(col("status") == "SUCCESS")
    .select("txn_id", "customer_id", "amount", "category", "txn_ts")
    .writeStream
    .format("memory")
    .queryName("once_result")
    .outputMode("append")
    .trigger(once=True)
    .option("checkpointLocation", os.path.join(CHECKPOINT, "once_query"))
    .start()
)

# awaitTermination() blocks until the query finishes (since once=True it will stop)
once_query.awaitTermination()

total = spark.sql("SELECT count(*) AS n FROM once_result").collect()[0][0]
print(f"once trigger processed {total} SUCCESS rows from all available files")
spark.sql("SELECT category, count(*) AS n FROM once_result GROUP BY category ORDER BY n DESC").show()

### Stateless Transformations — Filter, Select, Map

Most DataFrame operations work unchanged on streaming DataFrames. They are **stateless** — each row is processed independently with no memory of previous rows. Stateless operations always support `append` output mode.

| Operation | Streaming support | Notes |
|---|---|---|
| `filter` / `where` | Yes | |
| `select` / `selectExpr` | Yes | |
| `withColumn` | Yes | |
| `drop` | Yes | |
| `union` | Yes | |
| `groupBy` + `agg` | Yes (stateful) | Requires output mode `complete` or `update` |
| `window` | Yes (stateful) | Requires watermark for bounded state |
| `join` (stream ↔ static) | Yes | Broadcast join with a static DataFrame |
| `join` (stream ↔ stream) | Yes (stateful) | Requires watermarks on both sides |
| `sort` / `orderBy` | **Not supported** | Requires all data to be present |
| `distinct` | **Not supported** without watermark | Stateful deduplication needs a watermark |

### Writing to Files — Parquet Sink

The **file sink** writes micro-batch results to partitioned files. Each micro-batch produces one or more files in the output directory. Combined with `trigger(once=True)` or `trigger(availableNow=True)`, this replaces traditional batch ETL with streaming semantics (exactly-once, checkpoint-tracked).

File sinks only support **append** output mode. Each trigger appends new files — it cannot overwrite or update existing files.

For mutable sinks (update/delete), use **Delta Lake** as the sink format.

In [ ]:
txn_stream8 = (
    spark
    .readStream
    .schema(txn_schema)
    .json(STREAM_INPUT)
)

parquet_out = os.path.join(STREAM_OUTPUT, "txns_parquet")

parquet_query = (
    txn_stream8
    .filter(col("status") == "SUCCESS")
    .withColumn("year",  col("txn_ts").cast("date").cast("string").substr(1, 4))
    .withColumn("month", col("txn_ts").cast("date").cast("string").substr(6, 2))
    .writeStream
    .format("parquet")
    .option("path",              parquet_out)
    .option("checkpointLocation", os.path.join(CHECKPOINT, "parquet_query"))
    .partitionBy("year", "month")   # Hive-style partitioning: year=2024/month=01/
    .outputMode("append")
    .trigger(once=True)
    .start()
)

parquet_query.awaitTermination()

# Read back the written files
written = spark.read.parquet(parquet_out)
print(f"Rows written: {written.count()}")
written.select("txn_id", "customer_id", "amount", "category", "txn_ts").show(5, truncate=False)

### Monitoring Streaming Queries

The `StreamingQuery` object returned by `.start()` provides full observability:

| Property / Method | What it returns |
|---|---|
| `query.isActive` | `True` if the query is still running |
| `query.name` | The query name set by `.queryName()` |
| `query.id` | UUID — stable across restarts |
| `query.runId` | UUID — new each run |
| `query.status` | Dict: `{"message": "...", "isDataAvailable": bool, "isTriggerActive": bool}` |
| `query.lastProgress` | Dict with timing, input rates, state rows, watermark |
| `query.recentProgress` | List of the last 100 progress dicts |
| `query.awaitTermination(timeout)` | Block until stopped or timeout |
| `query.stop()` | Gracefully stop the query |
| `query.exception()` | Returns the exception if the query failed |
| `spark.streams.active` | List of all active queries in the session |

In [ ]:
monitoring_stream = (
    spark
    .readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
)

monitoring_query = (
    monitoring_stream
    .writeStream
    .format("memory")
    .queryName("monitoring_demo")
    .outputMode("append")
    .trigger(processingTime="2 seconds")
    .start()
)

time.sleep(6)  # let a few batches run

print("=== Query Status ===")
import pprint
pprint.pprint(monitoring_query.status)

print("\n=== Last Progress ===")
p = monitoring_query.lastProgress
if p:
    print(f"  batchId                : {p['batchId']}")
    print(f"  numInputRows           : {p['numInputRows']}")
    print(f"  inputRowsPerSecond     : {p['inputRowsPerSecond']}")
    print(f"  processedRowsPerSecond : {p['processedRowsPerSecond']}")
    print(f"  triggerExecution (ms)  : {p['triggerExecution']}")
    if p.get('stateOperators'):
        print(f"  stateOperators         : {p['stateOperators']}")

print(f"\nTotal active queries: {len(spark.streams.active)}")
monitoring_query.stop()

### Checkpointing and Fault Tolerance

Checkpointing is what makes Structured Streaming **exactly-once** and **restartable**. Without a checkpoint, if the Spark application crashes, the query must restart from scratch.

**What is stored in the checkpoint:**
- **Offsets** — the position in each source (file index, Kafka offset) processed in each batch
- **State store** — intermediate aggregation state (window counts, running totals)
- **Committed batch IDs** — which micro-batches have been fully written to the sink

```text
_checkpoints/my_query/
├── commits/        ← which batches are committed to the sink
├── offsets/        ← source positions for each batch
├── metadata        ← query metadata
└── state/          ← serialized state store (for stateful queries)
```

**Always set `checkpointLocation`** for any query you run in production. On restart, Spark reads the checkpoint and resumes exactly where it left off — no data is reprocessed or skipped.

**Checkpoint + sink compatibility**: For exactly-once delivery, the sink must also support idempotent writes or deduplication (Delta Lake, Kafka with transactions). The memory and console sinks are for development only and do not guarantee exactly-once.

In [ ]:
# Demonstrate checkpoint directory structure
import glob as _glob

checkpoint_path = os.path.join(CHECKPOINT, "once_query")
if os.path.exists(checkpoint_path):
    print("Checkpoint directory contents:")
    for root, dirs, files in os.walk(checkpoint_path):
        level = root.replace(checkpoint_path, "").count(os.sep)
        indent = "  " * level
        folder = os.path.basename(root)
        print(f"{indent}{folder}/")
        for f in files:
            subindent = "  " * (level + 1)
            print(f"{subindent}{f}")
else:
    print("No checkpoint found — run the once-trigger cell above first")

### Kafka Source (Reference — Requires Kafka)

Kafka is the most common production streaming source. The Spark-Kafka connector reads each Kafka message as a row with the columns below.

```python
kafka_stream = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "card_transactions")   # or subscribePattern for regex
    .option("startingOffsets", "latest")        # "earliest" or {"topic":{"0":offset}}
    .load()
)
```

Kafka row schema:

| Column | Type | Content |
|---|---|---|
| `key` | `BinaryType` | Message key (bytes) |
| `value` | `BinaryType` | Message payload (bytes) — **this is your data** |
| `topic` | `StringType` | Kafka topic name |
| `partition` | `IntegerType` | Kafka partition number |
| `offset` | `LongType` | Offset within the partition |
| `timestamp` | `TimestampType` | Kafka message timestamp |
| `timestampType` | `IntegerType` | 0 = CreateTime, 1 = LogAppendTime |

Deserializing the value:

```python
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StringType, DecimalType, TimestampType

txn_schema = StructType([...])

parsed = (
    kafka_stream
    .select(from_json(col("value").cast("string"), txn_schema).alias("data"))
    .select("data.*")
)
```

Writing back to Kafka:

```python
from pyspark.sql.functions import to_json, struct

result.select(
    col("txn_id").alias("key"),
    to_json(struct("*")).alias("value"),
).writeStream
 .format("kafka")
 .option("kafka.bootstrap.servers", "broker:9092")
 .option("topic", "fraud_alerts")
 .option("checkpointLocation", "/checkpoints/fraud_alerts")
 .start()
```

### Summary

**Structured Streaming** treats an incoming stream as an infinite table. You write the same DataFrame transformations you already know, and Spark runs them continuously as new data arrives in micro-batches.

**Sources**: rate (testing), file (JSON/Parquet/Delta), Kafka (production). Always provide a schema for file sources.

**Output modes**:
- `append` — new rows only (stateless ops)
- `update` — changed rows only (aggregations)
- `complete` — full result table (small aggregations)

**Triggers**:
- Default — as fast as possible
- `processingTime="30 seconds"` — fixed interval
- `once=True` — process all available data once, then stop
- `availableNow=True` — multi-batch version of `once`

**Checkpointing** saves offsets and state to durable storage. Always set `checkpointLocation` in production for exactly-once restartability.

**Monitoring** — the `StreamingQuery` object exposes `isActive`, `lastProgress`, `recentProgress`, `status`, and `stop()`.

> Sources, sinks, watermarking, and windowed aggregations are covered in depth in **notebook 16**. Arbitrary stateful processing, session windows, stream–stream joins, and deduplication are covered in **notebook 17**.


In [ ]:
# Cleanup scratch directories created by this notebook
for d in [STREAM_INPUT, STREAM_OUTPUT, CHECKPOINT]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed: {d}")

print("Cleanup complete")